In [ ]:
# Mount Google Drive and extract all text files in one go
from google.colab import drive

drive.mount('/content/drive')
import os, json, re
from pathlib import Path

# Install packages
!pip install -q PyPDF2 pdfplumber ebooklib beautifulsoup4 pylatexenc lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 60.2 MB/s eta 0:00:00


In [ ]:


# Import libraries
import PyPDF2, pdfplumber, ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
from pylatexenc.latex2text import LatexNodes2Text

# Setup paths
INPUT_DIR = "/content/drive/MyDrive/Cultural texts /Chinese"
OUTPUT_DIR = "/content/drive/MyDrive/Cultural texts /Chinese/extracted_texts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def extract_pdf(filepath):
    text_parts = []
    try:
        with pdfplumber.open(filepath) as pdf:
            for i, page in enumerate(pdf.pages, 1):
                if page_text := page.extract_text():
                    text_parts.append(f"\n{'='*50}\n Page {i}\n{'='*50}\n{page_text}")
        return "\n".join(text_parts)
    except:
        with open(filepath, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for i, page in enumerate(pdf_reader.pages, 1):
                if page_text := page.extract_text():
                    text_parts.append(f"\n{'='*50}\n Page {i}\n{'='*50}\n{page_text}")
        return "\n".join(text_parts)

def extract_epub(filepath):
    book = epub.read_epub(filepath)
    text_parts = []
    if title := book.get_metadata('DC', 'title'):
        text_parts.append(f"Title: {title[0][0]}\n")
    if author := book.get_metadata('DC', 'creator'):
        text_parts.append(f"Author: {author[0][0]}\n")
    text_parts.append("\n" + "="*60 + "\n\n")
    for item in book.get_items():
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
            soup = BeautifulSoup(item.get_content(), 'html.parser')
            for script in soup(["script", "style"]): script.decompose()
            text = soup.get_text()
            lines = (line.strip() for line in text.splitlines())
            text = '\n'.join(phrase.strip() for line in lines for phrase in line.split("  ") if phrase.strip())
            if text: text_parts.append(text + "\n\n")
    return "".join(text_parts)

def extract_latex(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        latex_content = f.read()
    try:
        return LatexNodes2Text().latex_to_text(latex_content)
    except:
        text = re.sub(r'%.*', '', latex_content)
        text = re.sub(r'\\(?:textbf|textit|emph|underline)\{([^}]*)\}', r'\1', text)
        text = re.sub(r'\\(?:section|subsection|chapter|title)\{([^}]*)\}', r'\n\n\1\n', text)
        text = re.sub(r'\\[a-zA-Z]+(\[.*?\])?(\{.*?\})*', '', text)
        text = re.sub(r'\$\$.*?\$\$', '[MATH]', text, flags=re.DOTALL)
        text = re.sub(r'\$.*?\$', '[MATH]', text)
        return re.sub(r'\n\s*\n', '\n\n', text).strip()

def extract_jsonl(filepath):
    text_parts = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                data = json.loads(line.strip())
                text_parts.append(f"\n--- Entry {line_num} ---")
                for key, value in data.items():
                    if isinstance(value, str):
                        text_parts.append(f"{key}: {value}")
                    elif isinstance(value, (list, dict)):
                        text_parts.append(f"{key}: {json.dumps(value, ensure_ascii=False)}")
            except: continue
    return "\n".join(text_parts)

# Extract all files
extractors = {'.pdf': extract_pdf, '.epub': extract_epub, '.tex': extract_latex, '.jsonl': extract_jsonl}
files = [f for f in Path(INPUT_DIR).iterdir() if f.is_file() and f.suffix.lower() in extractors]

print(f"🔍 Found {len(files)} files\n{'='*70}\n")
results = {'success': [], 'failed': []}

for i, filepath in enumerate(files, 1):
    print(f"[{i}/{len(files)}] {filepath.name}")
    try:
        text = extractors[filepath.suffix.lower()](str(filepath))
        output_path = Path(OUTPUT_DIR) / f"{filepath.stem}_extracted.txt"
        output_path.write_text(text, encoding='utf-8')
        results['success'].append(filepath.name)
        print(f"   ✅ Saved ({len(text):,} chars)\n")
    except Exception as e:
        results['failed'].append((filepath.name, str(e)))
        print(f"   ❌ Error: {e}\n")

print(f"\n{'='*70}\n✅ Success: {len(results['success'])}\n❌ Failed: {len(results['failed'])}\n📁 Output: {OUTPUT_DIR}")

🔍 Found 18 files

[1/18] dBET_Alpha_EssentialsofVinayaTraditionandCollectedTendaiTeachings_1995.pdf
   ✅ Saved (656,263 chars)

[2/18] dBET_T0262_LotusSutra_2007_0.pdf
   ✅ Saved (613,987 chars)

[3/18] dBET_Pratyutpanna_Surangama_1998_0.pdf
   ✅ Saved (437,900 chars)

[4/18] dBET_TiantaiLotusTexts_2013_0.pdf
   ✅ Saved (500,048 chars)

[5/18] dBET_T2186_ExpositoryCommentaryonVimalakirti_2012_0.pdf
   ✅ Saved (699,352 chars)

[6/18] dBET_ThreePureLandSutras_2003_0.pdf
   ✅ Saved (392,803 chars)

[7/18] dBET_T2185_ShotokusCommentaryonSrimala_2011.pdf
   ✅ Saved (457,605 chars)

[8/18] dokumen.pub_the-dream-of-the-red-chamber-9780804840965-2010005577.epub
   ✅ Saved (2,466,287 chars)

[9/18] dokumen.pub_xunzi-the-complete-text-course-booknbsped-9781400852550.pdf
   ✅ Saved (1,186,838 chars)

[10/18] dBET_DiamondNeedle_HuayanOneVehicle_SecretOfTheHeartSutra.pdf
   ✅ Saved (625,582 chars)

[11/18] dokumen.pub_to-live-ebooknbsped-0307429792-9780307429797.epub
   ✅ Saved (383,084 chars)

[12